# Quantum Error Analysis and Visualization

Este notebook procesa ejecuciones experimentales de simulaciones cuánticas, las compara con modelos analíticos teóricos y calcula métricas de error estándar (RMSE, MSE, R2).

**Correcciones Implementadas:**
1. **Estructura lógica y limpieza**: El código fue reorganizado de manera cronológica. Importaciones -> Modelos -> Procesamiento -> Visualización.
2. **Alineación de Datos (DataFrames)**: Se corrigió la lógica de creación de DataFrames para evitar índices duplicados y sobrescritura accidental (el error que eliminaba la columna `Noise`).
3. **Estandarización Independiente**: Se solucionó la fuga de datos/escalado aplicando `StandardScaler` a los datos con `Noise = 0.0` y `Noise = 9.0` de forma completamente aislada.
4. **Subplots de Facetas (Box Plots)**: Se mejoró la visualización de la dispersión de errores utilizando la función `facet_col` nativa de Plotly Express.

## 1. Libraries and Modules

In [ ]:
import pandas as pd
import numpy as np
from numpy import sin, cos, pi
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.metrics import r2_score as r2, mean_absolute_error as mae, mean_squared_error as mse, root_mean_squared_error as rmse
from sklearn.preprocessing import StandardScaler

## 2. Theoretical Models
Definimos el modelo matemático central que usaremos como base o "Ground Truth" para contrastar los resultados arrojados por el simulador.

In [ ]:
def analytical(theta: float, beta: float, alpha: float):
    return (
        cos(alpha)**4 + sin(alpha)**4 +
        2 * (sin(alpha)**2) * (cos(alpha)**2) * cos(theta) +
        cos(alpha+beta)**4 + sin(alpha+beta)**4 +
        2 * (sin(alpha+beta)**2) * (cos(alpha+beta)**2) * cos(theta)
    ) / 2

## 3. Data Loading and Metric Calculation
Extraemos los archivos CSV de las rutas indicadas, los procesamos, y extraemos las métricas RMSE, R2 y MSE comparándolos contra la matriz de la función analítica generada arriba.

In [ ]:
# Load filenames gracefully
try:
    reviewed_probability_filenames = pd.read_csv('./runs/filenames.csv', header=None).dropna(axis=1, how='all').values.flatten()
    noisy_probability_filenames = pd.read_csv('./runs/noisy_filenames.csv', header=None).dropna(axis=1, how='all').values.flatten()
except FileNotFoundError:
    print('Warning: File lists not found. Ensure the directory paths are correct.')
    reviewed_probability_filenames, noisy_probability_filenames = [], []

N_data = 20
theta_FSS = np.linspace(0, 2*pi, N_data)
beta_angle = np.linspace(0, pi, N_data)
alpha = pi/16
path = './runs/'

# Construir el modelo analítico (Ground Truth)
model = np.array([[analytical(theta=t, beta=b, alpha=alpha) for b in beta_angle] for t in theta_FSS])

noiseless_r2, noiseless_mse, noiseless_rmse = [], [], []
noisy_r2, noisy_mse, noisy_rmse = [], [], []

# Extracción de métricas para Noise = 0.0
for name in reviewed_probability_filenames:
    exp = pd.read_csv(path + name, header=None).dropna(axis=1, how='all').values    
    noiseless_r2.append(1 - r2(y_true=model, y_pred=exp))
    noiseless_mse.append(mse(y_true=model, y_pred=exp))
    noiseless_rmse.append(rmse(y_true=model, y_pred=exp))

# Extracción de métricas para Noise = 9.0
for name in noisy_probability_filenames:
    exp = pd.read_csv(path + name, header=None).dropna(axis=1, how='all').values    
    noisy_r2.append(1 - r2(y_true=model, y_pred=exp))
    noisy_mse.append(mse(y_true=model, y_pred=exp))
    noisy_rmse.append(rmse(y_true=model, y_pred=exp))

# Array de estados
N_states_list = [1.0E3, 5.0E3, 1.0E4, 1.5E4, 2.0E4, 2.5E4, 3.0E4, 3.5E4, 4.0E4, 4.5E4]

# Ensamblaje seguro de los DataFrames utilizando diccionarios. 
# NOTA: Se usa el slicing [:len()] para evitar caídas si un experimento no pudo cargar.
df_reviewed = pd.DataFrame({
    'Noise': 0.0, 
    'R2': noiseless_r2, 
    'RMSE': noiseless_rmse, 
    'MSE': noiseless_mse, 
    'N_states': N_states_list[:len(noiseless_rmse)]
})

df_noisy = pd.DataFrame({
    'Noise': 9.0, 
    'R2': noisy_r2, 
    'RMSE': noisy_rmse, 
    'MSE': noisy_mse, 
    'N_states': N_states_list[:len(noisy_rmse)]
})

results = pd.concat([df_reviewed, df_noisy], ignore_index=True)
results.head()

## 4. Independent Data Standardization
Debido a que el conjunto con Noise = 9.0 tiene un error varios órdenes de magnitud superior al Noise = 0.0, aplicar un único `StandardScaler` aplasta la desviación del experimento puro. 

Por este motivo, se enmascaran los datos por grupo de ruido, se escalan individualmente, y luego se unen con seguridad al DataFrame padre.

In [ ]:
# Crear un DF vacío compartiendo el mismo índice
scaled_results = pd.DataFrame(index=results.index, columns=['R2 scaled', 'RMSE scaled', 'MSE scaled'])

# Crear máscaras booleanas para cada experimento
mask_0 = results['Noise'] == 0.0
mask_9 = results['Noise'] == 9.0

scaler_0 = StandardScaler()
scaler_9 = StandardScaler()

# Entrenar y transformar SOLO si las máscaras contienen datos
if mask_0.any():
    scaled_results.loc[mask_0, ['R2 scaled', 'RMSE scaled', 'MSE scaled']] = scaler_0.fit_transform(results.loc[mask_0, ['R2', 'RMSE', 'MSE']])
if mask_9.any():
    scaled_results.loc[mask_9, ['R2 scaled', 'RMSE scaled', 'MSE scaled']] = scaler_9.fit_transform(results.loc[mask_9, ['R2', 'RMSE', 'MSE']])

# Asegurarse de que Plotly los acepte como punto flotante
scaled_results = scaled_results.astype(float)

# Concatenamos de forma limpia (¡Eliminé la línea redundante que borraba las columnas en tu código anterior!)
results_df = pd.concat([results, scaled_results], axis=1)
results_df.head()

## 5. Visualizing Error Trends
Observamos cómo decaen las métricas de error respecto al aumento de número de estados cuánticos (`N_states`).

In [ ]:
fig = px.line(
    results_df,
    x = 'N_states',
    y = 'MSE',
    color = 'Noise',
    markers = True
)

fig.update_traces(
    marker=dict(size=12, line=dict(width=1, color='DarkSlateGrey'))
)

fig.update_layout(
    title='Model Correlation: Error Decay vs Number of States',
    height = 500,
    width = 1000,
    font=dict(size=14)
)

fig.show()

## 6. Metric Dispersion (Box Plots with Facets)
Aquí solucionamos el problema de graficar ambos grupos. Al usar el parámetro **`facet_col='Noise'`**, PlotlyExpress automáticamente divide los datos escalados y te genera 2 subplots (uno al lado del otro), permitiéndote comparar las varianzas sin el problema del escalado disparejo.

In [ ]:
fig = px.box(
    results_df,
    y = ['MSE scaled', 'RMSE scaled', 'R2 scaled'],
    facet_col = 'Noise', # <- MAGIA: Esto crea los dos subplots automáticamente
    color = 'Noise',
    points = 'all',
    height = 550,
    width = 1100
)

fig.update_traces(
    marker=dict(size=8, line=dict(width=1, color='DarkSlateGrey')),
    selector=dict(type='box')
)

fig.update_layout(
    title='Dispersion of Standardized Error Metrics separated by Noise Level',
    font=dict(size=14),
    showlegend=True
)

fig.show()

## 7. 3D Surface Comparisons
Función parametrizada para visualizar la matriz teórica superpuesta y contrastada en 3D contra una matriz de resultados de un archivo particular.

In [ ]:
def plot_surface_from_csv(array_theoretical, array_experimental):
    fig = go.Figure()
    
    fig.add_trace(go.Surface(
        z=array_theoretical, 
        colorscale='Emrld',
        opacity=0.8,
        name='Theoretical',
        colorbar=dict(title='Theoretical', x=0.9)
    ))
    
    fig.add_trace(go.Surface(
        z=array_experimental, 
        opacity=0.8, 
        name='Experimental',
        colorbar=dict(title='Experimental', x=1.05)
    ))
    
    fig.update_layout(
        title='3D Surface Plot: Theoretical vs Experimental',
        autosize=False,
        width=900,
        height=700,
        margin=dict(l=65, r=50, b=65, t=90),
        scene=dict(
            xaxis_title='Phase Angle (θ)',
            yaxis_title='Beta Angle (β)',
            zaxis_title='Probability'
        ),
        font=dict(size=14) 
    )
    fig.show()

# Ejemplo de uso (Descomentar para usar con tus directorios válidos):
# expr = pd.read_csv('./runs/tu_archivo.csv', header=None).dropna(axis=1,how='all').values
# plot_surface_from_csv(model, expr)